# Luckfox Pico Zero (RV1106G3) — Project 7 All-in-One Vision Colab

Notebook này tạo trọn bộ **4 file** cho `proj_7_camera_lcd_mic_speaker/colab/`:
1. `yolov8n.rknn` (320×320 — hỗ trợ cả Segmentation `seg` hoặc Detection `det`)
2. `RetinaFace.rknn` (320×320 — tìm mặt & 5 điểm landmark)
3. `MobileFaceNet.rknn` (112×112 — trích xuất embedding nhận diện chủ nhân)
4. `rknn_vision_daemon` (Binary C++ Zero-Copy + `libjpeg-turbo` NEON + `--bin`, chạy YOLO trước rồi chỉ gọi RetinaFace + MobileFaceNet khi thấy `person`)

Chạy lần lượt **Cell 1 → Cell 2 → Cell 3**, tải `luckfox_proj7_deploy.zip` về giải nén vào `proj_7_camera_lcd_mic_speaker/colab/` rồi chạy `bash deploy.sh 192.168.1.3`.


In [ ]:
%%bash
# Cell 1: Cai moi truong + Toolchain RV1106 + Convert 3 model (yolov8n, RetinaFace, MobileFaceNet)
set -e
cd /content
PY=/opt/rknn_env/bin/python
MODEL_TYPE="seg"  # Chon "seg" (YOLOv8n-Seg co Masking) hoac "det" (YOLOv8n chi Box)
IMG_SIZE=320      # Chon 320 (nhanh nhat cho RV1106) hoac 416 / 640

# 1. Cai Python 3.11 + rknn-toolkit2 + ultralytics
if ! "$PY" -c "import rknn.api, ultralytics, onnx, cv2" 2>/dev/null; then
  echo "==> [1/6] Cai dat rknn-toolkit2 + ultralytics..."
  pip install -q uv
  [ -x "$PY" ] || uv venv --python 3.11 /opt/rknn_env
  uv pip install --python "$PY" rknn-toolkit2 "setuptools<81" "numpy<2" opencv-python ultralytics
else
  echo "==> [1/6] /opt/rknn_env da san sang."
fi

# 2. Tai nhanh Toolchain uClibc RV1106 bang Sparse-Checkout
TC_DIR="/content/luckfox-pico/tools/linux/toolchain/arm-rockchip830-linux-uclibcgnueabihf"
if [ ! -d "$TC_DIR" ]; then
  echo "==> [2/6] Tai Toolchain RV1106..."
  rm -rf /content/luckfox-pico
  git clone --depth 1 --filter=blob:none --sparse https://github.com/LuckfoxTECH/luckfox-pico.git /content/luckfox-pico
  git -C /content/luckfox-pico sparse-checkout set tools/linux/toolchain/arm-rockchip830-linux-uclibcgnueabihf
else
  echo "==> [2/6] Toolchain RV1106 da co san."
fi

# 3. Clone rknn_model_zoo
if [ ! -d "/content/rknn_model_zoo" ]; then
  echo "==> [3/6] Clone rknn_model_zoo..."
  git clone --depth 1 https://github.com/airockchip/rknn_model_zoo.git /content/rknn_model_zoo
else
  echo "==> [3/6] rknn_model_zoo da co san."
fi

# 4. Convert Model 1: YOLOv8n (Seg/Det)
echo "==> [4/6] Convert yolov8n.rknn ($MODEL_TYPE, ${IMG_SIZE}x${IMG_SIZE})..."
MODEL_TYPE="$MODEL_TYPE" IMG_SIZE="$IMG_SIZE" "$PY" - << 'EOF'
import os, types, torch
from ultralytics import YOLO
from rknn.api import RKNN

model_type = os.environ.get("MODEL_TYPE", "seg")
img_size = int(os.environ.get("IMG_SIZE", "320"))
pt_name = "yolov8n-seg.pt" if model_type == "seg" else "yolov8n.pt"
onnx_path = f"/content/yolov8n_{model_type}_{img_size}.onnx"
dataset_txt = "/content/rknn_model_zoo/datasets/COCO/coco_subset_20.txt"

yolo = YOLO(pt_name)
m = yolo.model.eval()
head = m.model[-1]
is_seg = hasattr(head, "proto")

def rknn_head_forward(self, x):
    outs = []
    for i in range(self.nl):
        box = self.cv2[i](x[i])
        cls = torch.sigmoid(self.cv3[i](x[i]))
        score_sum = torch.clamp(cls.sum(dim=1, keepdim=True), 0.0, 1.0)
        outs.extend([box, cls, score_sum])
        if is_seg:
            outs.append(self.cv4[i](x[i]))
    if is_seg:
        outs.append(self.proto(x[0]))
    return tuple(outs)

head.forward = types.MethodType(rknn_head_forward, head)
dummy = torch.zeros(1, 3, img_size, img_size)
with torch.no_grad():
    sample_outs = m(dummy)
torch.onnx.export(
    m, dummy, onnx_path, opset_version=12, input_names=["images"],
    output_names=[f"out_{i}" for i in range(len(sample_outs))],
)

rknn = RKNN(verbose=False)
rknn.config(mean_values=[[0, 0, 0]], std_values=[[255, 255, 255]], target_platform="rv1106")
assert rknn.load_onnx(model=onnx_path) == 0
assert rknn.build(do_quantization=True, dataset=dataset_txt) == 0
assert rknn.export_rknn("/content/yolov8n.rknn") == 0
rknn.release()
print("Convert OK -> /content/yolov8n.rknn")
EOF

# 5. Convert Model 2: RetinaFace.rknn
if [ ! -f "/content/RetinaFace.rknn" ]; then
  echo "==> [5/6] Tai & convert RetinaFace.rknn..."
  cd /content/rknn_model_zoo/examples/RetinaFace/model
  if [ ! -f "RetinaFace_mobile320.onnx" ]; then
    timeout 60 bash download_model.sh
  fi
  cd /content/rknn_model_zoo/examples/RetinaFace/python
  "$PY" convert.py ../model/RetinaFace_mobile320.onnx rv1106
  cp ../model/RetinaFace.rknn /content/RetinaFace.rknn
else
  echo "==> [5/6] /content/RetinaFace.rknn da co san."
fi

# 6. Convert Model 3: MobileFaceNet.rknn (InsightFace buffalo_sc w600k_mbf.onnx)
if [ ! -f "/content/MobileFaceNet.rknn" ]; then
  echo "==> [6/6] Tai & convert MobileFaceNet.rknn..."
  cd /content
  if [ ! -f "/content/w600k_mbf.onnx" ]; then
    curl -fSL --connect-timeout 15 --max-time 90 -o /content/buffalo_sc.zip \
      "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_sc.zip"
    unzip -oj /content/buffalo_sc.zip w600k_mbf.onnx -d /content
  fi
  "$PY" - << 'EOF'
import glob, os, cv2, onnx
from rknn.api import RKNN

calib_dir = "/content/calib_112"
os.makedirs(calib_dir, exist_ok=True)
srcs = []
rf_test = "/content/rknn_model_zoo/examples/RetinaFace/model/test.jpg"
if os.path.isfile(rf_test):
    srcs.append(rf_test)
srcs.extend(sorted(glob.glob("/content/rknn_model_zoo/datasets/COCO/*.jpg"))[:11])
paths = []
for idx, p in enumerate(srcs):
    img = cv2.imread(p)
    if img is None:
        continue
    out_p = os.path.join(calib_dir, f"c_{idx:02d}.jpg")
    cv2.imwrite(out_p, cv2.resize(img, (112, 112)))
    paths.append(out_p)
assert len(paths) > 0
ds_txt = "/content/calib_112/dataset.txt"
with open(ds_txt, "w") as f:
    f.write("\n".join(paths) + "\n")

inp_name = onnx.load("/content/w600k_mbf.onnx").graph.input[0].name
rknn = RKNN(verbose=False)
rknn.config(
    mean_values=[[127.5, 127.5, 127.5]],
    std_values=[[127.5, 127.5, 127.5]],
    target_platform="rv1106",
    quantized_dtype="asymmetric_quantized-8",
)
assert rknn.load_onnx(model="/content/w600k_mbf.onnx", inputs=[inp_name], input_size_list=[[1, 3, 112, 112]]) == 0
assert rknn.build(do_quantization=True, dataset=ds_txt) == 0
assert rknn.export_rknn("/content/MobileFaceNet.rknn") == 0
rknn.release()
print(f"Convert OK ({len(paths)} calib imgs) -> /content/MobileFaceNet.rknn")
EOF
else
  echo "==> [6/6] /content/MobileFaceNet.rknn da co san."
fi

echo "==> Hoan tat 3 model RKNN:"
ls -lh /content/yolov8n.rknn /content/RetinaFace.rknn /content/MobileFaceNet.rknn

In [ ]:
%%bash
# Cell 2: Bien dich rknn_vision_daemon (3-model Zero-Copy + libjpeg-turbo NEON + --bin)
set -e
rm -f /content/rknn_vision_daemon
cat << 'EOF' > /content/vision_daemon.cc
/*
 * Unified 3-Model Zero-Copy Vision Daemon for Luckfox Pico Zero (RV1106G3)
 *   1) YOLOv8n (Det 6/9-out or Seg 10/13-out)
 *   2) RetinaFace (320x320) -> runs only when YOLO detects a person
 *   3) MobileFaceNet (112x112) -> L2-normalized embedding per detected face
 */
#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <string>
#include <vector>

#include "rknn_api.h"
#ifdef USE_TURBOJPEG
#include "turbojpeg.h"
#else
#define STBI_ONLY_JPEG
#define STB_IMAGE_IMPLEMENTATION
#include "stb_image.h"
#endif

struct DetBox {
  float x1, y1, x2, y2;
  float score;
  int cls_id;
  int branch;
  int grid_r;
  int grid_c;
};

struct BranchIdx {
  int box_idx;
  int cls_idx;
  int sum_idx;
  int seg_idx;
};

struct FaceBox {
  float x1, y1, x2, y2;
  float score;
  float lm[10];
  std::vector<float> emb;
};

struct Prior {
  float cx, cy, sx, sy;
};

static const char* COCO_NAMES[80] = {
    "person","bicycle","car","motorcycle","airplane","bus","train","truck","boat","traffic light",
    "fire hydrant","stop sign","parking meter","bench","bird","cat","dog","horse","sheep","cow",
    "elephant","bear","zebra","giraffe","backpack","umbrella","handbag","tie","suitcase","frisbee",
    "skis","snowboard","sports ball","kite","baseball bat","baseball glove","skateboard","surfboard","tennis racket","bottle",
    "wine glass","cup","fork","knife","spoon","bowl","banana","apple","sandwich","orange",
    "broccoli","carrot","hot dog","pizza","donut","cake","chair","couch","potted plant","bed",
    "dining table","toilet","tv","laptop","mouse","remote","keyboard","cell phone","microwave","oven",
    "toaster","sink","refrigerator","book","clock","vase","scissors","teddy bear","hair drier","toothbrush"
};

static std::string base64_encode(const uint8_t* data, size_t len) {
  static const char* B64 =
      "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/";
  std::string out;
  out.reserve(((len + 2) / 3) * 4);
  for (size_t i = 0; i < len; i += 3) {
    uint32_t v = (uint32_t)data[i] << 16;
    if (i + 1 < len) v |= (uint32_t)data[i + 1] << 8;
    if (i + 2 < len) v |= (uint32_t)data[i + 2];
    out.push_back(B64[(v >> 18) & 0x3F]);
    out.push_back(B64[(v >> 12) & 0x3F]);
    out.push_back((i + 1 < len) ? B64[(v >> 6) & 0x3F] : '=');
    out.push_back((i + 2 < len) ? B64[v & 0x3F] : '=');
  }
  return out;
}

static inline float fp16_to_fp32(uint16_t h) {
  uint32_t sign = (uint32_t)(h & 0x8000) << 16;
  uint32_t exp = (h & 0x7C00) >> 10;
  uint32_t mant = (h & 0x03FF);
  uint32_t f;
  if (exp == 0) {
    if (mant == 0) {
      f = sign;
    } else {
      exp = 1;
      while ((mant & 0x0400) == 0) {
        mant <<= 1;
        exp--;
      }
      mant &= 0x03FF;
      f = sign | ((exp + 112) << 23) | (mant << 13);
    }
  } else if (exp == 31) {
    f = sign | 0x7F800000 | (mant << 13);
  } else {
    f = sign | ((exp + 112) << 23) | (mant << 13);
  }
  float res;
  memcpy(&res, &f, sizeof(res));
  return res;
}

static inline size_t elem_size_of(rknn_tensor_type t) {
  if (t == RKNN_TENSOR_FLOAT32) return 4;
  if (t == RKNN_TENSOR_FLOAT16) return 2;
  return 1;
}

struct OutView {
  int h;
  int w;
  int c;
  int c2;
  int stride_w;
  rknn_tensor_format fmt;

  void init(const rknn_tensor_attr& a) {
    fmt = a.fmt;
    c2 = 1;
    size_t esz = elem_size_of(a.type);
    if (fmt == RKNN_TENSOR_NC1HWC2 && a.n_dims >= 5) {
      c2 = (int)a.dims[4];
      c = (int)a.dims[1] * c2;
      h = (int)a.dims[2];
      w = (int)a.dims[3];
      stride_w = (a.w_stride >= (uint32_t)w) ? (int)a.w_stride : w;
    } else if (fmt == RKNN_TENSOR_NHWC && a.n_dims >= 4) {
      h = (int)a.dims[1];
      w = (int)a.dims[2];
      c = (int)a.dims[3];
      if (a.w_stride > (uint32_t)w &&
          (size_t)h * a.w_stride * c * esz <= a.size_with_stride &&
          a.size_with_stride > (size_t)h * w * c * esz) {
        stride_w = (int)a.w_stride;
      } else {
        stride_w = w;
      }
    } else {
      c = (int)a.dims[1];
      h = (int)a.dims[2];
      w = (int)a.dims[3];
      stride_w = (a.w_stride > (uint32_t)w &&
                  (size_t)c * h * a.w_stride * esz <= a.size_with_stride)
                     ? (int)a.w_stride
                     : w;
    }
  }

  inline int offset(int r, int col, int ch) const {
    if (fmt == RKNN_TENSOR_NHWC) {
      return (r * stride_w + col) * c + ch;
    }
    if (fmt == RKNN_TENSOR_NC1HWC2) {
      int c1_idx = ch / c2;
      int c2_idx = ch % c2;
      return (((c1_idx * h + r) * stride_w) + col) * c2 + c2_idx;
    }
    return (ch * h + r) * stride_w + col;
  }
};

struct RknnZeroCopy {
  rknn_context ctx;
  rknn_input_output_num io_num;
  rknn_tensor_attr in_attr;
  rknn_tensor_mem* in_mem;
  std::vector<rknn_tensor_attr> out_attrs;
  std::vector<rknn_tensor_mem*> out_mems;
  std::vector<OutView> out_views;
  int model_w;
  int model_h;
  int in_stride;

  RknnZeroCopy() : ctx(0), in_mem(nullptr), model_w(0), model_h(0), in_stride(0) {
    memset(&io_num, 0, sizeof(io_num));
    memset(&in_attr, 0, sizeof(in_attr));
  }

  bool init(const char* path, bool prefer_nhwc_out) {
    FILE* fp = fopen(path, "rb");
    if (!fp) {
      fprintf(stderr, "Cannot open model %s\n", path);
      return false;
    }
    fseek(fp, 0, SEEK_END);
    size_t sz = ftell(fp);
    fseek(fp, 0, SEEK_SET);
    std::vector<uint8_t> buf(sz);
    if (fread(buf.data(), 1, sz, fp) != sz) {
      fclose(fp);
      return false;
    }
    fclose(fp);

    int ret = rknn_init(&ctx, buf.data(), sz, 0, nullptr);
    if (ret < 0) {
      fprintf(stderr, "rknn_init %s fail %d\n", path, ret);
      ctx = 0;
      return false;
    }

    rknn_query(ctx, RKNN_QUERY_IN_OUT_NUM, &io_num, sizeof(io_num));
    memset(&in_attr, 0, sizeof(in_attr));
    in_attr.index = 0;
    rknn_query(ctx, RKNN_QUERY_NATIVE_INPUT_ATTR, &in_attr, sizeof(in_attr));
    in_attr.type = RKNN_TENSOR_UINT8;
    in_attr.fmt = RKNN_TENSOR_NHWC;
    model_h = (int)in_attr.dims[1];
    model_w = (int)in_attr.dims[2];
    in_stride = (in_attr.w_stride >= (uint32_t)model_w) ? (int)in_attr.w_stride : model_w;

    in_mem = rknn_create_mem(ctx, in_attr.size_with_stride);
    if (!in_mem || rknn_set_io_mem(ctx, in_mem, &in_attr) < 0) {
      fprintf(stderr, "rknn_set_io_mem in fail %s\n", path);
      return false;
    }

    out_attrs.resize(io_num.n_output);
    out_mems.resize(io_num.n_output, nullptr);
    out_views.resize(io_num.n_output);
    for (uint32_t i = 0; i < io_num.n_output; i++) {
      memset(&out_attrs[i], 0, sizeof(rknn_tensor_attr));
      out_attrs[i].index = i;
      ret = -1;
      if (prefer_nhwc_out) {
        ret = rknn_query(ctx, RKNN_QUERY_NATIVE_NHWC_OUTPUT_ATTR, &out_attrs[i], sizeof(rknn_tensor_attr));
      }
      if (ret < 0) {
        memset(&out_attrs[i], 0, sizeof(rknn_tensor_attr));
        out_attrs[i].index = i;
        ret = rknn_query(ctx, RKNN_QUERY_NATIVE_OUTPUT_ATTR, &out_attrs[i], sizeof(rknn_tensor_attr));
        if (ret < 0) return false;
      }
      out_mems[i] = rknn_create_mem(ctx, out_attrs[i].size_with_stride);
      if (!out_mems[i] || rknn_set_io_mem(ctx, out_mems[i], &out_attrs[i]) < 0) return false;
      out_views[i].init(out_attrs[i]);
    }
    return true;
  }

  int run() {
    rknn_mem_sync(ctx, in_mem, RKNN_MEMORY_SYNC_TO_DEVICE);
    int ret = rknn_run(ctx, nullptr);
    for (uint32_t i = 0; i < io_num.n_output; i++) {
      rknn_mem_sync(ctx, out_mems[i], RKNN_MEMORY_SYNC_FROM_DEVICE);
    }
    return ret;
  }

  void destroy() {
    if (in_mem) {
      rknn_destroy_mem(ctx, in_mem);
      in_mem = nullptr;
    }
    for (size_t i = 0; i < out_mems.size(); i++) {
      if (out_mems[i]) {
        rknn_destroy_mem(ctx, out_mems[i]);
        out_mems[i] = nullptr;
      }
    }
    if (ctx) {
      rknn_destroy(ctx);
      ctx = 0;
    }
  }
};

static inline float get_val(const void* data, int idx, const rknn_tensor_attr& attr) {
  if (attr.type == RKNN_TENSOR_INT8)
    return ((float)((const int8_t*)data)[idx] - (float)attr.zp) * attr.scale;
  if (attr.type == RKNN_TENSOR_UINT8)
    return ((float)((const uint8_t*)data)[idx] - (float)attr.zp) * attr.scale;
  if (attr.type == RKNN_TENSOR_FLOAT16)
    return fp16_to_fp32(((const uint16_t*)data)[idx]);
  return ((const float*)data)[idx];
}

static inline int32_t quant_thresh_i8(float thresh, const rknn_tensor_attr& attr) {
  if (attr.scale <= 0.0f) return -128;
  float q = floorf(thresh / attr.scale + (float)attr.zp);
  if (q < -128.0f) return -128;
  if (q > 127.0f) return 127;
  return (int32_t)q;
}

template <typename T>
static float box_iou(const T& a, const T& b) {
  float xx1 = std::max(a.x1, b.x1), yy1 = std::max(a.y1, b.y1);
  float xx2 = std::min(a.x2, b.x2), yy2 = std::min(a.y2, b.y2);
  float inter = std::max(0.0f, xx2 - xx1) * std::max(0.0f, yy2 - yy1);
  float area_a = std::max(0.0f, a.x2 - a.x1) * std::max(0.0f, a.y2 - a.y1);
  float area_b = std::max(0.0f, b.x2 - b.x1) * std::max(0.0f, b.y2 - b.y1);
  return inter / (area_a + area_b - inter + 1e-6f);
}

static void letterbox_rgb(const uint8_t* src, int sw, int sh,
                          uint8_t* dst, int dw, int dh, int dst_stride, uint8_t pad_val,
                          float& scale, int& pad_x, int& pad_y) {
  scale = std::min((float)dw / sw, (float)dh / sh);
  int nw = (int)(sw * scale), nh = (int)(sh * scale);
  pad_x = (dw - nw) / 2;
  pad_y = (dh - nh) / 2;
  memset(dst, pad_val, (size_t)dst_stride * dh * 3);

  if (nw == sw && nh == sh) {
    size_t row_bytes = (size_t)sw * 3;
    for (int y = 0; y < sh; y++) {
      memcpy(dst + ((y + pad_y) * dst_stride + pad_x) * 3, src + y * row_bytes, row_bytes);
    }
    return;
  }

  std::vector<int> x_lut(nw);
  std::vector<int> y_lut(nh);
  for (int x = 0; x < nw; x++) x_lut[x] = std::min(sw - 1, (int)(x / scale));
  for (int y = 0; y < nh; y++) y_lut[y] = std::min(sh - 1, (int)(y / scale));

  for (int y = 0; y < nh; y++) {
    uint8_t* drow = dst + ((y + pad_y) * dst_stride + pad_x) * 3;
    const uint8_t* srow = src + y_lut[y] * sw * 3;
    for (int x = 0; x < nw; x++) {
      const uint8_t* sp = srow + x_lut[x] * 3;
      drow[x * 3 + 0] = sp[0];
      drow[x * 3 + 1] = sp[1];
      drow[x * 3 + 2] = sp[2];
    }
  }
}

static void decode_yolo_branch(
    int branch_id,
    const void* box_buf, const rknn_tensor_attr& box_attr, const OutView& box_view,
    const void* cls_buf, const rknn_tensor_attr& cls_attr, const OutView& cls_view,
    const void* sum_buf, const rknn_tensor_attr* sum_attr, const OutView* sum_view,
    int model_h, int sw, int sh, float scale, int pad_x, int pad_y,
    float conf_thresh, std::vector<DetBox>& out) {
  int grid_h = box_view.h;
  int grid_w = box_view.w;
  if (grid_h <= 0 || grid_w <= 0) return;

  int stride = model_h / grid_h;
  int dfl_len = box_view.c / 4;
  if (dfl_len <= 0 || dfl_len > 32) return;
  int num_cls = cls_view.c;

  bool sum_i8 = (sum_buf && sum_attr && sum_attr->type == RKNN_TENSOR_INT8);
  bool cls_i8 = (cls_attr.type == RKNN_TENSOR_INT8);
  int32_t sum_q_th = sum_i8 ? quant_thresh_i8(conf_thresh, *sum_attr) : -128;
  int32_t cls_q_th = cls_i8 ? quant_thresh_i8(conf_thresh, cls_attr) : -128;

  const int8_t* sum_p = (const int8_t*)sum_buf;
  const int8_t* cls_p = (const int8_t*)cls_buf;

  for (int r = 0; r < grid_h; r++) {
    for (int c = 0; c < grid_w; c++) {
      if (sum_buf && sum_attr && sum_view) {
        int soff = sum_view->offset(r, c, 0);
        if (sum_i8) {
          if (sum_p[soff] < sum_q_th) continue;
        } else {
          if (get_val(sum_buf, soff, *sum_attr) < conf_thresh) continue;
        }
      }

      int best_cls = -1;
      float best_score = 0.0f;
      if (cls_i8) {
        int32_t max_q = cls_q_th - 1;
        if (cls_view.fmt == RKNN_TENSOR_NHWC) {
          const int8_t* cell_cls = cls_p + (r * cls_view.stride_w + c) * num_cls;
          for (int k = 0; k < num_cls; k++) {
            int32_t q = cell_cls[k];
            if (q > max_q) {
              max_q = q;
              best_cls = k;
            }
          }
        } else {
          for (int k = 0; k < num_cls; k++) {
            int32_t q = cls_p[cls_view.offset(r, c, k)];
            if (q > max_q) {
              max_q = q;
              best_cls = k;
            }
          }
        }
        if (best_cls < 0) continue;
        best_score = ((float)max_q - (float)cls_attr.zp) * cls_attr.scale;
      } else {
        best_score = conf_thresh;
        for (int k = 0; k < num_cls; k++) {
          float v = get_val(cls_buf, cls_view.offset(r, c, k), cls_attr);
          if (v >= best_score) {
            best_score = v;
            best_cls = k;
          }
        }
        if (best_cls < 0) continue;
      }

      if (best_score < conf_thresh) continue;

      float ltrb[4] = {0.0f, 0.0f, 0.0f, 0.0f};
      for (int b = 0; b < 4; b++) {
        float probs[32];
        float max_v = -1e9f;
        int base = b * dfl_len;
        for (int d = 0; d < dfl_len; d++) {
          float v = get_val(box_buf, box_view.offset(r, c, base + d), box_attr);
          probs[d] = v;
          if (v > max_v) max_v = v;
        }
        float sum_e = 0.0f, acc = 0.0f;
        for (int d = 0; d < dfl_len; d++) {
          float e = expf(probs[d] - max_v);
          probs[d] = e;
          sum_e += e;
        }
        for (int d = 0; d < dfl_len; d++) {
          acc += (probs[d] / sum_e) * d;
        }
        ltrb[b] = acc * stride;
      }

      float cx = (c + 0.5f) * stride;
      float cy = (r + 0.5f) * stride;
      float x1 = std::max(0.0f, std::min((float)sw, ((cx - ltrb[0]) - pad_x) / scale));
      float y1 = std::max(0.0f, std::min((float)sh, ((cy - ltrb[1]) - pad_y) / scale));
      float x2 = std::max(0.0f, std::min((float)sw, ((cx + ltrb[2]) - pad_x) / scale));
      float y2 = std::max(0.0f, std::min((float)sh, ((cy + ltrb[3]) - pad_y) / scale));
      if (x2 <= x1 || y2 <= y1) continue;

      DetBox db;
      db.x1 = x1;
      db.y1 = y1;
      db.x2 = x2;
      db.y2 = y2;
      db.score = best_score;
      db.cls_id = best_cls;
      db.branch = branch_id;
      db.grid_r = r;
      db.grid_c = c;
      out.push_back(db);
    }
  }
}

struct MaskOut {
  int w;
  int h;
  float rx1, ry1, rx2, ry2;
  std::string b64;
};

static bool build_mask_b64(
    const DetBox& d,
    const void* seg_buf, const rknn_tensor_attr& seg_attr, const OutView& seg_view,
    const void* proto_buf, const rknn_tensor_attr& proto_attr, const OutView& proto_view,
    int model_w, int model_h, float scale, int pad_x, int pad_y,
    MaskOut& out) {
  int nc = std::min(32, proto_view.c);
  if (nc <= 0 || proto_view.w <= 0 || proto_view.h <= 0) return false;

  float mc[32];
  float mc_sum = 0.0f;
  for (int k = 0; k < nc; k++) {
    mc[k] = get_val(seg_buf, seg_view.offset(d.grid_r, d.grid_c, k), seg_attr);
    mc_sum += mc[k];
  }

  float proto_sx = (float)proto_view.w / (float)model_w;
  float proto_sy = (float)proto_view.h / (float)model_h;
  float mx1 = d.x1 * scale + (float)pad_x;
  float my1 = d.y1 * scale + (float)pad_y;
  float mx2 = d.x2 * scale + (float)pad_x;
  float my2 = d.y2 * scale + (float)pad_y;

  int px1 = std::max(0, (int)floorf(mx1 * proto_sx));
  int py1 = std::max(0, (int)floorf(my1 * proto_sy));
  int px2 = std::min(proto_view.w, (int)ceilf(mx2 * proto_sx));
  int py2 = std::min(proto_view.h, (int)ceilf(my2 * proto_sy));
  int mw = px2 - px1;
  int mh = py2 - py1;
  if (mw <= 0 || mh <= 0) return false;

  out.w = mw;
  out.h = mh;
  out.rx1 = ((float)px1 / proto_sx - (float)pad_x) / scale;
  out.ry1 = ((float)py1 / proto_sy - (float)pad_y) / scale;
  out.rx2 = ((float)px2 / proto_sx - (float)pad_x) / scale;
  out.ry2 = ((float)py2 / proto_sy - (float)pad_y) / scale;

  std::vector<uint8_t> bits(((size_t)mw * mh + 7) / 8, 0);
  bool proto_i8 = (proto_attr.type == RKNN_TENSOR_INT8 && proto_attr.scale > 0.0f);
  const int8_t* p_i8 = (const int8_t*)proto_buf;
  float zp_mc_sum = (float)proto_attr.zp * mc_sum;

  size_t bit_idx = 0;
  for (int py = py1; py < py2; py++) {
    for (int px = px1; px < px2; px++, bit_idx++) {
      bool on = false;
      if (proto_i8) {
        float acc = 0.0f;
        if (proto_view.fmt == RKNN_TENSOR_NHWC) {
          const int8_t* cell = p_i8 + (py * proto_view.stride_w + px) * proto_view.c;
          for (int k = 0; k < nc; k++) acc += mc[k] * (float)cell[k];
        } else {
          for (int k = 0; k < nc; k++) {
            acc += mc[k] * (float)p_i8[proto_view.offset(py, px, k)];
          }
        }
        on = (acc > zp_mc_sum);
      } else {
        float acc = 0.0f;
        for (int k = 0; k < nc; k++) {
          acc += mc[k] * get_val(proto_buf, proto_view.offset(py, px, k), proto_attr);
        }
        on = (acc > 0.0f);
      }
      if (on) {
        bits[bit_idx >> 3] |= (uint8_t)(1u << (bit_idx & 7));
      }
    }
  }

  out.b64 = base64_encode(bits.data(), bits.size());
  return true;
}

static std::vector<Prior> generate_priors(int img_w, int img_h) {
  std::vector<Prior> priors;
  const int steps[3] = {8, 16, 32};
  const int min_sizes[3][2] = {{16, 32}, {64, 128}, {256, 512}};
  for (int k = 0; k < 3; k++) {
    int fw = (int)ceil((float)img_w / steps[k]);
    int fh = (int)ceil((float)img_h / steps[k]);
    for (int i = 0; i < fh; i++) {
      for (int j = 0; j < fw; j++) {
        for (int m = 0; m < 2; m++) {
          Prior p;
          p.cx = (j + 0.5f) * steps[k] / img_w;
          p.cy = (i + 0.5f) * steps[k] / img_h;
          p.sx = (float)min_sizes[k][m] / img_w;
          p.sy = (float)min_sizes[k][m] / img_h;
          priors.push_back(p);
        }
      }
    }
  }
  return priors;
}

static void align_face_112(const uint8_t* src, int sw, int sh,
                           const float lm[10], uint8_t* dst, int dw, int dh, int dst_stride) {
  static const float ref[5][2] = {
      {38.2946f, 51.6963f}, {73.5318f, 51.5014f},
      {56.0252f, 71.7366f}, {41.5493f, 92.3655f}, {70.7299f, 92.2041f}};
  float scale_x = (float)dw / 112.0f, scale_y = (float)dh / 112.0f;
  float mxd = 0, myd = 0, mxs = 0, mys = 0;
  for (int i = 0; i < 5; i++) {
    mxd += ref[i][0] * scale_x;
    myd += ref[i][1] * scale_y;
    mxs += lm[i * 2];
    mys += lm[i * 2 + 1];
  }
  mxd /= 5.0f;
  myd /= 5.0f;
  mxs /= 5.0f;
  mys /= 5.0f;

  float num_a = 0, num_b = 0, denom = 0;
  for (int i = 0; i < 5; i++) {
    float dx = ref[i][0] * scale_x - mxd;
    float dy = ref[i][1] * scale_y - myd;
    float sx = lm[i * 2] - mxs;
    float sy = lm[i * 2 + 1] - mys;
    num_a += dx * sx + dy * sy;
    num_b += dx * sy - dy * sx;
    denom += dx * dx + dy * dy;
  }
  float a = num_a / (denom + 1e-6f);
  float b = num_b / (denom + 1e-6f);
  float tx = mxs - (a * mxd - b * myd);
  float ty = mys - (b * mxd + a * myd);

  for (int y = 0; y < dh; y++) {
    uint8_t* out_row = dst + y * dst_stride * 3;
    for (int x = 0; x < dw; x++) {
      float sx = a * x - b * y + tx;
      float sy = b * x + a * y + ty;
      uint8_t* out = out_row + x * 3;
      int x0 = (int)floorf(sx), y0 = (int)floorf(sy);
      if (x0 < 0 || y0 < 0 || x0 + 1 >= sw || y0 + 1 >= sh) {
        out[0] = out[1] = out[2] = 0;
        continue;
      }
      float fx = sx - x0, fy = sy - y0;
      const uint8_t* p00 = src + (y0 * sw + x0) * 3;
      const uint8_t* p10 = p00 + 3;
      const uint8_t* p01 = p00 + sw * 3;
      const uint8_t* p11 = p01 + 3;
      for (int c = 0; c < 3; c++) {
        float v = (1 - fx) * (1 - fy) * p00[c] + fx * (1 - fy) * p10[c] +
                  (1 - fx) * fy * p01[c] + fx * fy * p11[c];
        out[c] = (uint8_t)std::min(255.0f, std::max(0.0f, v));
      }
    }
  }
}

static void dump_tft_rgb565(const uint8_t* rgb, int sw, int sh,
                            int tw, int th, const char* path,
                            std::vector<uint8_t>& buf) {
  if (!rgb || sw <= 0 || sh <= 0 || tw <= 0 || th <= 0 || !path || !path[0]) return;
  size_t need = (size_t)tw * (size_t)th * 2;
  if (buf.size() != need) buf.resize(need);
  uint8_t* dst = buf.data();

  if (sw == tw && sh == th) {
    size_t n_pix = (size_t)tw * (size_t)th;
    for (size_t i = 0; i < n_pix; i++) {
      const uint8_t* p = rgb + i * 3;
      uint16_t c = (uint16_t)(((p[0] & 0xF8) << 8) | ((p[1] & 0xFC) << 3) | (p[2] >> 3));
      dst[i * 2 + 0] = (uint8_t)(c >> 8);
      dst[i * 2 + 1] = (uint8_t)(c & 0xFF);
    }
  } else {
    for (int y = 0; y < th; y++) {
      int sy = (y * sh) / th;
      const uint8_t* row = rgb + (size_t)sy * (size_t)sw * 3;
      uint8_t* out_row = dst + (size_t)y * (size_t)tw * 2;
      for (int x = 0; x < tw; x++) {
        int sx = (x * sw) / tw;
        const uint8_t* p = row + (size_t)sx * 3;
        uint16_t c = (uint16_t)(((p[0] & 0xF8) << 8) | ((p[1] & 0xFC) << 3) | (p[2] >> 3));
        out_row[x * 2 + 0] = (uint8_t)(c >> 8);
        out_row[x * 2 + 1] = (uint8_t)(c & 0xFF);
      }
    }
  }

  std::string tmp = std::string(path) + ".tmp";
  FILE* fp = fopen(tmp.c_str(), "wb");
  if (!fp) return;
  size_t wr = fwrite(dst, 1, need, fp);
  fclose(fp);
  if (wr == need) {
    rename(tmp.c_str(), path);
  }
}

static bool read_exact(FILE* fp, void* dst, size_t n) {
  uint8_t* p = (uint8_t*)dst;
  size_t got = 0;
  while (got < n) {
    size_t r = fread(p + got, 1, n - got, fp);
    if (r == 0) return false;
    got += r;
  }
  return true;
}

int main(int argc, char** argv) {
  if (argc < 2) {
    fprintf(stderr, "Usage: %s <yolov8n.rknn> [--bin]\n", argv[0]);
    return 1;
  }
  bool bin_mode = (argc >= 3 && strcmp(argv[2], "--bin") == 0);

  std::string yolo_path = argv[1];
  std::string dir = ".";
  size_t slash = yolo_path.find_last_of('/');
  if (slash != std::string::npos) dir = yolo_path.substr(0, slash);
  std::string fdet_path = dir + "/RetinaFace.rknn";
  std::string frec_path = dir + "/MobileFaceNet.rknn";

  RknnZeroCopy yolo, fdet, frec;
  if (!yolo.init(yolo_path.c_str(), true)) return 1;

  bool has_face = false;
  if (fdet.init(fdet_path.c_str(), false) && frec.init(frec_path.c_str(), false)) {
    has_face = true;
  } else {
    fdet.destroy();
    frec.destroy();
    fprintf(stderr, "Face models optional or missing -> running YOLO only\n");
  }

  int proto_idx = -1;
  for (uint32_t i = 0; i < yolo.io_num.n_output; i++) {
    if (yolo.out_views[i].h == yolo.model_h / 4 &&
        yolo.out_views[i].w == yolo.model_w / 4 &&
        yolo.out_views[i].c == 32) {
      proto_idx = (int)i;
      break;
    }
  }
  bool has_seg = (proto_idx >= 0);

  BranchIdx branches[3];
  for (int k = 0; k < 3; k++) {
    branches[k].box_idx = -1;
    branches[k].cls_idx = -1;
    branches[k].sum_idx = -1;
    branches[k].seg_idx = -1;
  }
  const int strides[3] = {8, 16, 32};
  for (uint32_t i = 0; i < yolo.io_num.n_output; i++) {
    if ((int)i == proto_idx) continue;
    int b = -1;
    for (int k = 0; k < 3; k++) {
      if (yolo.out_views[i].h == yolo.model_h / strides[k] &&
          yolo.out_views[i].w == yolo.model_w / strides[k]) {
        b = k;
        break;
      }
    }
    if (b < 0) continue;
    if (yolo.out_views[i].c == 64 && branches[b].box_idx < 0) {
      branches[b].box_idx = (int)i;
    } else if (yolo.out_views[i].c == 1 && branches[b].sum_idx < 0) {
      branches[b].sum_idx = (int)i;
    } else if (has_seg && yolo.out_views[i].c == 32 && branches[b].seg_idx < 0) {
      branches[b].seg_idx = (int)i;
    } else if (branches[b].cls_idx < 0) {
      branches[b].cls_idx = (int)i;
    }
  }

  std::vector<Prior> face_priors;
  int emb_dim = 0;
  if (has_face) {
    face_priors = generate_priors(fdet.model_w, fdet.model_h);
    emb_dim = (int)frec.out_attrs[0].n_elems;
  }

  fprintf(stderr, "Vision daemon ready: yolo=%dx%d (%s), has_face=%s\n",
          yolo.model_w, yolo.model_h, has_seg ? "SEG" : "DET",
          has_face ? "true" : "false");

#ifdef USE_TURBOJPEG
  tjhandle tj = tjInitDecompress();
  if (!tj) {
    fprintf(stderr, "tjInitDecompress fail\n");
    return 1;
  }
  const char* decoder_name = "turbo";
  std::vector<uint8_t> rgb_buf;
#else
  const char* decoder_name = "stb";
#endif

  printf("%s\n", bin_mode ? "READY_BIN" : "READY");
  fflush(stdout);

  const float yolo_conf = 0.20f;
  const float nms_thresh = 0.45f;
  const char* fe_env = getenv("FACE_EVERY");
  int face_every = fe_env ? atoi(fe_env) : 5;
  if (face_every < 1) face_every = 1;
  const char* te_env = getenv("TFT_EVERY");
  int tft_every = te_env ? atoi(te_env) : 3;
  if (tft_every < 0) tft_every = 0;
  const char* tw_env = getenv("TFT_W");
  const char* th_env = getenv("TFT_H");
  int tft_w = (tw_env && atoi(tw_env) > 0) ? atoi(tw_env) : 320;
  int tft_h = (th_env && atoi(th_env) > 0) ? atoi(th_env) : 240;
  const char* tp_env = getenv("TFT_RGB565_PATH");
  std::string tft_path = (tp_env && tp_env[0]) ? tp_env : "/dev/shm/tft_bg.rgb565";
  uint64_t frame_idx = 0;
  uint64_t person_frames = 0;
  std::vector<uint8_t> tft_buf;
  std::vector<uint8_t> jpg_buf;
  char line[512];

  while (true) {
    int sw = 0, sh = 0;
    uint8_t* img = nullptr;
    uint32_t frame_sz = 0;

    auto td0 = std::chrono::steady_clock::now();
    if (bin_mode) {
      uint8_t hdr[4];
      if (!read_exact(stdin, hdr, 4)) break;
      frame_sz = ((uint32_t)hdr[0] << 24) | ((uint32_t)hdr[1] << 16) |
                 ((uint32_t)hdr[2] << 8) | (uint32_t)hdr[3];
      if (frame_sz == 0 || frame_sz > 8u * 1024u * 1024u) {
        printf("{\"err\":\"bad frame size\",\"dets\":[],\"faces\":[]}\n");
        fflush(stdout);
        break;
      }
      if (jpg_buf.size() < frame_sz) jpg_buf.resize(frame_sz);
      if (!read_exact(stdin, jpg_buf.data(), frame_sz)) break;
    } else {
      if (!fgets(line, sizeof(line), stdin)) break;
      size_t len = strlen(line);
      while (len > 0 && (line[len - 1] == '\n' || line[len - 1] == '\r')) line[--len] = '\0';
      if (len == 0) continue;
#ifdef USE_TURBOJPEG
      FILE* jf = fopen(line, "rb");
      if (jf) {
        fseek(jf, 0, SEEK_END);
        long fsz = ftell(jf);
        fseek(jf, 0, SEEK_SET);
        if (fsz > 0 && fsz <= 8L * 1024L * 1024L) {
          frame_sz = (uint32_t)fsz;
          if (jpg_buf.size() < frame_sz) jpg_buf.resize(frame_sz);
          if (fread(jpg_buf.data(), 1, frame_sz, jf) != frame_sz) frame_sz = 0;
        }
        fclose(jf);
      }
#endif
    }

#ifdef USE_TURBOJPEG
    if (frame_sz > 0) {
      int subsamp = 0, cs = 0;
      if (tjDecompressHeader3(tj, jpg_buf.data(), (unsigned long)frame_sz,
                              &sw, &sh, &subsamp, &cs) == 0 &&
          sw > 0 && sh > 0) {
        size_t need = (size_t)sw * (size_t)sh * 3;
        if (rgb_buf.size() < need) rgb_buf.resize(need);
        if (tjDecompress2(tj, jpg_buf.data(), (unsigned long)frame_sz,
                          rgb_buf.data(), sw, 0, sh, TJPF_RGB, TJFLAG_FASTDCT) == 0) {
          img = rgb_buf.data();
        }
      }
    }
#else
    int sc = 0;
    if (bin_mode) {
      img = stbi_load_from_memory(jpg_buf.data(), (int)frame_sz, &sw, &sh, &sc, 3);
    } else {
      img = stbi_load(line, &sw, &sh, &sc, 3);
    }
#endif
    auto td1 = std::chrono::steady_clock::now();
    double dec_ms = std::chrono::duration<double, std::milli>(td1 - td0).count();

    if (!img) {
      printf("{\"err\":\"decode fail\",\"decoder\":\"%s\",\"dec_ms\":%.2f,\"dets\":[],\"faces\":[]}\n",
             decoder_name, dec_ms);
      fflush(stdout);
      continue;
    }

    if (tft_every > 0 && (frame_idx % (uint64_t)tft_every) == 0) {
      dump_tft_rgb565(img, sw, sh, tft_w, tft_h, tft_path.c_str(), tft_buf);
    }
    frame_idx++;

    // 1. Run YOLOv8n
    float y_scale = 1.0f;
    int y_pad_x = 0, y_pad_y = 0;
    letterbox_rgb(img, sw, sh, (uint8_t*)yolo.in_mem->virt_addr,
                  yolo.model_w, yolo.model_h, yolo.in_stride, 114, y_scale, y_pad_x, y_pad_y);

    auto ty0 = std::chrono::steady_clock::now();
    int yret = yolo.run();
    auto ty1 = std::chrono::steady_clock::now();
    double yolo_ms = std::chrono::duration<double, std::milli>(ty1 - ty0).count();

    auto tp0 = std::chrono::steady_clock::now();
    std::vector<DetBox> cand;
    if (yret >= 0) {
      for (int b = 0; b < 3; b++) {
        int box_idx = branches[b].box_idx;
        int cls_idx = branches[b].cls_idx;
        int sum_idx = branches[b].sum_idx;
        if (box_idx < 0 || cls_idx < 0) continue;
        decode_yolo_branch(
            b,
            yolo.out_mems[box_idx]->virt_addr, yolo.out_attrs[box_idx], yolo.out_views[box_idx],
            yolo.out_mems[cls_idx]->virt_addr, yolo.out_attrs[cls_idx], yolo.out_views[cls_idx],
            sum_idx >= 0 ? yolo.out_mems[sum_idx]->virt_addr : nullptr,
            sum_idx >= 0 ? &yolo.out_attrs[sum_idx] : nullptr,
            sum_idx >= 0 ? &yolo.out_views[sum_idx] : nullptr,
            yolo.model_h, sw, sh, y_scale, y_pad_x, y_pad_y, yolo_conf, cand);
      }
    }

    std::sort(cand.begin(), cand.end(),
              [](const DetBox& a, const DetBox& b) { return a.score > b.score; });
    std::vector<DetBox> kept;
    bool has_person = false;
    for (const auto& c : cand) {
      bool sup = false;
      for (const auto& k : kept) {
        if (k.cls_id == c.cls_id && box_iou(c, k) > nms_thresh) {
          sup = true;
          break;
        }
      }
      if (!sup) {
        if (c.cls_id == 0 && c.score >= 0.30f) has_person = true;
        kept.push_back(c);
      }
      if (kept.size() >= 25) break;
    }

    std::vector<MaskOut> masks(kept.size());
    std::vector<bool> mask_ok(kept.size(), false);
    if (has_seg) {
      size_t limit = std::min<size_t>(kept.size(), 10);
      for (size_t i = 0; i < limit; i++) {
        const auto& d = kept[i];
        int seg_idx = (d.branch >= 0 && d.branch < 3) ? branches[d.branch].seg_idx : -1;
        if (seg_idx < 0) continue;
        mask_ok[i] = build_mask_b64(
            d,
            yolo.out_mems[seg_idx]->virt_addr, yolo.out_attrs[seg_idx], yolo.out_views[seg_idx],
            yolo.out_mems[proto_idx]->virt_addr, yolo.out_attrs[proto_idx], yolo.out_views[proto_idx],
            yolo.model_w, yolo.model_h, y_scale, y_pad_x, y_pad_y, masks[i]);
      }
    }
    auto tp1 = std::chrono::steady_clock::now();
    double post_ms = std::chrono::duration<double, std::milli>(tp1 - tp0).count();

    // 2. Conditional Face Detection + Embedding (throttled every FACE_EVERY person frames)
    double fdet_ms = 0.0, frec_ms = 0.0;
    std::vector<FaceBox> kept_faces;
    bool face_ran = false;
    if (has_face && has_person) {
      bool run_now = ((person_frames % (uint64_t)face_every) == 0);
      person_frames++;
      if (run_now) {
        face_ran = true;
        float f_scale = 1.0f;
        int f_pad_x = 0, f_pad_y = 0;
        letterbox_rgb(img, sw, sh, (uint8_t*)fdet.in_mem->virt_addr,
                      fdet.model_w, fdet.model_h, fdet.in_stride, 0, f_scale, f_pad_x, f_pad_y);

        auto tf0 = std::chrono::steady_clock::now();
        int fret = fdet.run();
        auto tf1 = std::chrono::steady_clock::now();
        fdet_ms = std::chrono::duration<double, std::milli>(tf1 - tf0).count();

        std::vector<FaceBox> f_cand;
        if (fret >= 0) {
          for (int i = 0; i < (int)face_priors.size(); i++) {
            float score = get_val(fdet.out_mems[1]->virt_addr, i * 2 + 1, fdet.out_attrs[1]);
            if (score < 0.55f) continue;

            float l0 = get_val(fdet.out_mems[0]->virt_addr, i * 4 + 0, fdet.out_attrs[0]);
            float l1 = get_val(fdet.out_mems[0]->virt_addr, i * 4 + 1, fdet.out_attrs[0]);
            float l2 = get_val(fdet.out_mems[0]->virt_addr, i * 4 + 2, fdet.out_attrs[0]);
            float l3 = get_val(fdet.out_mems[0]->virt_addr, i * 4 + 3, fdet.out_attrs[0]);
            const Prior& p = face_priors[i];
            float cx = p.cx + l0 * 0.1f * p.sx, cy = p.cy + l1 * 0.1f * p.sy;
            float bw = p.sx * expf(l2 * 0.2f), bh = p.sy * expf(l3 * 0.2f);

            FaceBox fb;
            fb.x1 = ((cx - bw * 0.5f) * fdet.model_w - f_pad_x) / f_scale;
            fb.y1 = ((cy - bh * 0.5f) * fdet.model_h - f_pad_y) / f_scale;
            fb.x2 = ((cx + bw * 0.5f) * fdet.model_w - f_pad_x) / f_scale;
            fb.y2 = ((cy + bh * 0.5f) * fdet.model_h - f_pad_y) / f_scale;
            fb.score = score;
            for (int k = 0; k < 5; k++) {
              float lx = get_val(fdet.out_mems[2]->virt_addr, i * 10 + k * 2 + 0, fdet.out_attrs[2]);
              float ly = get_val(fdet.out_mems[2]->virt_addr, i * 10 + k * 2 + 1, fdet.out_attrs[2]);
              fb.lm[k * 2 + 0] = ((p.cx + lx * 0.1f * p.sx) * fdet.model_w - f_pad_x) / f_scale;
              fb.lm[k * 2 + 1] = ((p.cy + ly * 0.1f * p.sy) * fdet.model_h - f_pad_y) / f_scale;
            }
            f_cand.push_back(fb);
          }
        }

        std::sort(f_cand.begin(), f_cand.end(),
                  [](const FaceBox& a, const FaceBox& b) { return a.score > b.score; });
        for (const auto& fc : f_cand) {
          bool sup = false;
          for (const auto& kf : kept_faces) {
            if (box_iou(fc, kf) > 0.40f) {
              sup = true;
              break;
            }
          }
          if (!sup) kept_faces.push_back(fc);
          if (kept_faces.size() >= 2) break;
        }

        for (size_t i = 0; i < kept_faces.size(); i++) {
          FaceBox& fb = kept_faces[i];
          align_face_112(img, sw, sh, fb.lm, (uint8_t*)frec.in_mem->virt_addr,
                         frec.model_w, frec.model_h, frec.in_stride);
          auto tr0 = std::chrono::steady_clock::now();
          int rret = frec.run();
          auto tr1 = std::chrono::steady_clock::now();
          frec_ms += std::chrono::duration<double, std::milli>(tr1 - tr0).count();

          if (rret >= 0 && emb_dim > 0) {
            fb.emb.resize(emb_dim);
            float norm = 0.0f;
            for (int d = 0; d < emb_dim; d++) {
              float v = get_val(frec.out_mems[0]->virt_addr, d, frec.out_attrs[0]);
              fb.emb[d] = v;
              norm += v * v;
            }
            norm = sqrtf(norm) + 1e-6f;
            for (int d = 0; d < emb_dim; d++) fb.emb[d] /= norm;
          }
        }
      }
    } else {
      person_frames = 0;
    }

#ifndef USE_TURBOJPEG
    stbi_image_free(img);
#endif

    double total_npu = yolo_ms + fdet_ms + frec_ms;
    printf("{\"npu_ms\":%.2f,\"yolo_ms\":%.2f,\"fdet_ms\":%.2f,\"frec_ms\":%.2f,"
           "\"dec_ms\":%.2f,\"post_ms\":%.2f,\"decoder\":\"%s\","
           "\"has_seg\":%s,\"has_face\":%s,\"face_ran\":%s,\"model_w\":%d,\"model_h\":%d,"
           "\"w\":%d,\"h\":%d,\"dets\":[",
           total_npu, yolo_ms, fdet_ms, frec_ms, dec_ms, post_ms, decoder_name,
           has_seg ? "true" : "false", has_face ? "true" : "false",
           face_ran ? "true" : "false",
           yolo.model_w, yolo.model_h, sw, sh);
    for (size_t i = 0; i < kept.size(); i++) {
      const auto& d = kept[i];
      const char* lbl = (d.cls_id >= 0 && d.cls_id < 80) ? COCO_NAMES[d.cls_id] : "obj";
      if (i > 0) printf(",");
      printf("{\"label\":\"%s\",\"cls\":%d,\"score\":%.3f,\"box\":[%.1f,%.1f,%.1f,%.1f]",
             lbl, d.cls_id, d.score, d.x1, d.y1, d.x2, d.y2);
      if (mask_ok[i]) {
        const auto& m = masks[i];
        printf(",\"mask\":{\"w\":%d,\"h\":%d,\"rect\":[%.1f,%.1f,%.1f,%.1f],\"b64\":\"%s\"}",
               m.w, m.h, m.rx1, m.ry1, m.rx2, m.ry2, m.b64.c_str());
      }
      printf("}");
    }
    printf("],\"faces\":[");
    for (size_t i = 0; i < kept_faces.size(); i++) {
      const auto& f = kept_faces[i];
      if (i > 0) printf(",");
      printf("{\"score\":%.3f,\"box\":[%.1f,%.1f,%.1f,%.1f],\"lm\":[%.1f,%.1f,%.1f,%.1f,%.1f,%.1f,%.1f,%.1f,%.1f,%.1f],\"emb\":[",
             f.score, f.x1, f.y1, f.x2, f.y2,
             f.lm[0], f.lm[1], f.lm[2], f.lm[3], f.lm[4],
             f.lm[5], f.lm[6], f.lm[7], f.lm[8], f.lm[9]);
      for (size_t d = 0; d < f.emb.size(); d++) {
        if (d > 0) printf(",");
        printf("%.4f", f.emb[d]);
      }
      printf("]}");
    }
    printf("]}\n");
    fflush(stdout);
  }

#ifdef USE_TURBOJPEG
  tjDestroy(tj);
#endif
  yolo.destroy();
  fdet.destroy();
  frec.destroy();
  return 0;
}
EOF

CXX=/content/luckfox-pico/tools/linux/toolchain/arm-rockchip830-linux-uclibcgnueabihf/bin/arm-rockchip830-linux-uclibcgnueabihf-g++
RKNN_INC=/content/rknn_model_zoo/3rdparty/rknpu2/include
RKNN_LIB=/content/rknn_model_zoo/3rdparty/rknpu2/Linux/armhf-uclibc
STB_INC=/content/rknn_model_zoo/3rdparty/stb_image

TJ_LIB=""
for p in \
  /content/rknn_model_zoo/3rdparty/jpeg_turbo/Linux/armhf_uclibc/libturbojpeg.a \
  /content/rknn_model_zoo/3rdparty/jpeg_turbo/Linux/armhf-uclibc/libturbojpeg.a; do
  if [ -f "$p" ]; then TJ_LIB="$p"; break; fi
done
if [ -z "$TJ_LIB" ] && [ -d /content/rknn_model_zoo/3rdparty/jpeg_turbo ]; then
  TJ_LIB="$(find /content/rknn_model_zoo/3rdparty/jpeg_turbo -name 'libturbojpeg.a' | grep -i uclibc | head -n 1 || true)"
fi
TJ_HDR="$(find /content/rknn_model_zoo/3rdparty/jpeg_turbo -name 'turbojpeg.h' 2>/dev/null | head -n 1 || true)"
TJ_INC=""
if [ -n "$TJ_HDR" ]; then TJ_INC="$(dirname "$TJ_HDR")"; fi

BUILT=0
if [ -n "$TJ_LIB" ] && [ -n "$TJ_INC" ]; then
  echo "==> Thu bien dich voi libjpeg-turbo NEON: $TJ_LIB"
  if "$CXX" -O3 -mfpu=neon -mfloat-abi=hard -std=c++11 -DUSE_TURBOJPEG /content/vision_daemon.cc \
      -I"$RKNN_INC" -I"$TJ_INC" "$TJ_LIB" \
      -L"$RKNN_LIB" -lrknnmrt -lpthread -lm \
      -o /content/rknn_vision_daemon; then
    BUILT=1
    echo "==> Build OK voi decoder: libjpeg-turbo (NEON)"
  else
    echo "==> Link libjpeg-turbo loi, chuyen sang stb_image fallback..."
  fi
fi

if [ "$BUILT" -eq 0 ]; then
  "$CXX" -O3 -mfpu=neon -mfloat-abi=hard -std=c++11 /content/vision_daemon.cc \
    -I"$RKNN_INC" -I"$STB_INC" \
    -L"$RKNN_LIB" -lrknnmrt -lpthread -lm \
    -o /content/rknn_vision_daemon
  echo "==> Build OK voi decoder: stb_image"
fi

grep -qa "has_seg" /content/rknn_vision_daemon && grep -qa "has_face" /content/rknn_vision_daemon && grep -qa "face_ran" /content/rknn_vision_daemon && echo "==> VERIFIED: rknn_vision_daemon ho tro YOLO Seg/Det + RetinaFace + MobileFaceNet + TFT RGB565!"
ls -lh /content/rknn_vision_daemon

In [ ]:
# Cell 3: Dong goi tat ca 3 model + rknn_vision_daemon thanh 1 file zip de tai ve
import os
import zipfile
from google.colab import files

files_to_download = [
    "/content/yolov8n.rknn",
    "/content/RetinaFace.rknn",
    "/content/MobileFaceNet.rknn",
    "/content/rknn_vision_daemon",
]
zip_filename = "/content/luckfox_proj7_deploy.zip"

with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for f in files_to_download:
        assert os.path.exists(f), f"Thieu file: {f}"
        zipf.write(f, os.path.basename(f))
        print(f"Da them: {os.path.basename(f)} ({os.path.getsize(f):,} bytes)")

print("Dang tai xuong:", zip_filename)
files.download(zip_filename)
